# MongoDB Assignment

### Task 1: Create and Insert Data
**Objective**: Learn how to create a new database and insert a single document into a collection.

**Description**: In this task, you will create a database called `school` and a collection named `students`. You will then insert a single student record into the collection. This teaches you the basic `use` command and `insertOne` method.

In [2]:
import pymongo

# Create a connection to the local MongoDB server running on Windows
# Default connection string for local instance is mongodb://localhost:27017/
client = pymongo.MongoClient("mongodb://localhost:27017/")

# Verify connection by listing existing database names
print("Connected successfully! Current databases:", client.list_database_names())

Connected successfully! Current databases: ['admin', 'config', 'local', 'my_sooq']


In [3]:
# 1. Access/Create the database named 'school'
db = client["school"]

# 2. Access/Create the collection named 'students'
students_collection = db["students"]

# 3. Define a single student document to insert
new_student = {
    "name": "Amran",
    "age": 25,
    "major": "Data Engineering"
}

# 4. Insert the single document into the collection
insert_result = students_collection.insert_one(new_student)

# 5. Print confirmation along with the automatically generated Unique ID (_id)
print("Task 1 Completed Successfully!")
print("Inserted Document ID:", insert_result.inserted_id)

Task 1 Completed Successfully!
Inserted Document ID: 6a314ccadb09899cbc0c970b


### Task 2: Insert Multiple Documents
**Objective**: Learn to insert multiple documents in a single command.

**Description**: You'll insert two additional student records into the `students` collection. This helps you understand how `insertMany` works to add multiple entries at once.

In [4]:
# 1. Define a list of multiple student documents to insert
multiple_students = [
    {
        "name": "Noah",
        "age": 21,
        "major": "Math"
    },
    {
        "name": "Emma",
        "age": 20,
        "major": "Biology"
    }
]

# 2. Insert the list of documents into the 'students' collection
batch_insert_result = students_collection.insert_many(multiple_students)

# 3. Print confirmation and the list of unique IDs assigned to the inserted documents
print("Task 2 Completed Successfully!")
print("Inserted Documents IDs:", batch_insert_result.inserted_ids)

Task 2 Completed Successfully!
Inserted Documents IDs: [ObjectId('6a314d22db09899cbc0c970c'), ObjectId('6a314d22db09899cbc0c970d')]


### Task 3: Querying Documents
**Objective**: Practice querying documents using specific conditions.

**Description**: You will write queries to find students who major in Math and those who are older than 20. This task introduces filtering documents with fields and comparison operators.

In [5]:
# 1. Query students who major in Math
math_query = {"major": "Math"}
math_students = list(students_collection.find(math_query))

print("--- Students Majoring in Math ---")
for student in math_students:
    print(student)

# 2. Query students who are older than 20 using the $gt operator
age_query = {"age": {"$gt": 20}}
older_students = list(students_collection.find(age_query))

print("\n--- Students Older Than 20 ---")
for student in older_students:
    print(student)

--- Students Majoring in Math ---
{'_id': ObjectId('6a314d22db09899cbc0c970c'), 'name': 'Noah', 'age': 21, 'major': 'Math'}

--- Students Older Than 20 ---
{'_id': ObjectId('6a314ccadb09899cbc0c970b'), 'name': 'Amran', 'age': 25, 'major': 'Data Engineering'}
{'_id': ObjectId('6a314d22db09899cbc0c970c'), 'name': 'Noah', 'age': 21, 'major': 'Math'}


### Task 4: Update a Document
**Objective**: Learn how to update a document’s field.

**Description**: Update the `major` of the student named Noah to "Computer Science". This demonstrates the use of the `$set` operator in `updateOne`.

In [6]:
# 1. Define the filter to locate the student named Noah
student_filter = {"name": "Noah"}

# 2. Define the update operation to change the major using $set
update_operation = {"$set": {"major": "Computer Science"}}

# 3. Apply the update to a single matching document
update_result = students_collection.update_one(student_filter, update_operation)

# 4. Print confirmation and the number of modified documents
print("Task 4 Completed Successfully!")
print("Documents Matched:", update_result.matched_count)
print("Documents Modified:", update_result.modified_count)

Task 4 Completed Successfully!
Documents Matched: 1
Documents Modified: 1


### Task 5: Remove a Field
**Objective**: Learn how to remove a field from a document.

**Description**: Remove the `age` field from Emma's document. This task introduces the `$unset` operator to delete specific fields from documents.

In [7]:
# 1. Define the filter to locate Emma's document
emma_filter = {"name": "Emma"}

# 2. Define the update operation to remove the 'age' field using $unset
# The value provided to the field in $unset can be an empty string "" or 1
unset_operation = {"$unset": {"age": ""}}

# 3. Execute the update command
unset_result = students_collection.update_one(emma_filter, unset_operation)

# 4. Print confirmation and verification steps
print("Task 5 Completed Successfully!")
print("Documents Matched:", unset_result.matched_count)
print("Documents Modified:", unset_result.modified_count)

# 5. Fetch Emma's document to visually verify the field is gone
print("\nEmma's updated record:")
print(students_collection.find_one({"name": "Emma"}))

Task 5 Completed Successfully!
Documents Matched: 1
Documents Modified: 1

Emma's updated record:
{'_id': ObjectId('6a314d22db09899cbc0c970d'), 'name': 'Emma', 'major': 'Biology'}


### Task 6: Use Operators for Advanced Queries
**Objective**: Use range conditions to query documents.

**Description**: Write a query to find all students whose ages are between 20 and 22 inclusive. This involves using `$gte` and `$lte` operators together.

In [8]:
# 1. Define the range query using $gte and $lte operators
range_query = {
    "age": {
        "$gte": 20, 
        "$lte": 22
    }
}

# 2. Execute the find query and convert the cursor result to a list
students_in_range = list(students_collection.find(range_query))

# 3. Print the results
print("Task 6 Completed Successfully!")
print("Students with ages between 20 and 22 inclusive:")
for student in students_in_range:
    print(student)

Task 6 Completed Successfully!
Students with ages between 20 and 22 inclusive:
{'_id': ObjectId('6a314d22db09899cbc0c970c'), 'name': 'Noah', 'age': 21, 'major': 'Computer Science'}


### Task 7: Use Aggregation Pipeline
**Objective**: Group documents and perform counts.

**Description**: Count the number of students per major. This will demonstrate MongoDB's `aggregate` function and `$group` stage.

In [9]:
# 1. Define the aggregation pipeline with a $group stage
aggregation_pipeline = [
    {
        "$group": {
            "_id": "$major",                      # Group by the 'major' field
            "student_count": {"$sum": 1}          # Increment the counter by 1 for each matching document
        }
    }
]

# 2. Execute the aggregation pipeline on the collection
aggregation_results = list(students_collection.aggregate(aggregation_pipeline))

# 3. Print the results clearly
print("Task 7 Completed Successfully!")
print("Number of students per major:")
for result in aggregation_results:
    print(f"Major: {result['_id']} | Total Students: {result['student_count']}")

Task 7 Completed Successfully!
Number of students per major:
Major: Biology | Total Students: 1
Major: Data Engineering | Total Students: 1
Major: Computer Science | Total Students: 1


### Task 8: Field Comparison with $expr
**Objective**: Compare two fields in a document.

**Description**: Insert students with `marks` and `passingMarks`. Then find students whose `marks` are less than `passingMarks` using `$expr`.

In [10]:
# 1. Insert new student documents with 'marks' and 'passingMarks' fields
new_records = [
    {"name": "Sara", "marks": 85, "passingMarks": 90},  # Marks (85) < Passing Marks (90) -> Should be found
    {"name": "John", "marks": 95, "passingMarks": 90}   # Marks (95) > Passing Marks (90) -> Should be ignored
]
students_collection.insert_many(new_records)
print("New student records inserted for validation.")

# 2. Query documents where 'marks' field is less than 'passingMarks' field using $expr
comparison_query = {
    "$expr": {
        "$lt": ["$marks", "$passingMarks"]  # Compares the value of $marks with $passingMarks
    }
}

# 3. Execute the query
failed_students = list(students_collection.find(comparison_query))

# 4. Print the results
print("\nTask 8 Completed Successfully!")
print("Students whose marks are less than passing marks:")
for student in failed_students:
    print(student)

New student records inserted for validation.

Task 8 Completed Successfully!
Students whose marks are less than passing marks:
{'_id': ObjectId('6a314e37db09899cbc0c970e'), 'name': 'Sara', 'marks': 85, 'passingMarks': 90}


### Task 9: Use Array Queries
**Objective**: Query and update array fields.

**Description**: Insert a student with an array of courses. Then add a new course to the list and retrieve all students enrolled in a specific course. Learn how to use `$push` to update arrays and how to match documents based on array values.

In [11]:
# 1. Insert a new student record with an array of initial courses
student_with_array = {
    "name": "Omar",
    "courses": ["Math", "Physics"]
}
students_collection.insert_one(student_with_array)
print("Initial record for Omar with courses array inserted.")

# 2. Update the document to add a new course ('Chemistry') using $push
array_update_filter = {"name": "Omar"}
push_operation = {"$push": {"courses": "Chemistry"}}

students_collection.update_one(array_update_filter, push_operation)
print("Course 'Chemistry' successfully pushed to Omar's array.")

# 3. Retrieve all students enrolled in the specific course 'Chemistry'
array_query = {"courses": "Chemistry"}
chemistry_students = list(students_collection.find(array_query))

# 4. Print the final results
print("\nTask 9 Completed Successfully!")
print("Students enrolled in Chemistry:")
for student in chemistry_students:
    print(student)

Initial record for Omar with courses array inserted.
Course 'Chemistry' successfully pushed to Omar's array.

Task 9 Completed Successfully!
Students enrolled in Chemistry:
{'_id': ObjectId('6a314e5edb09899cbc0c9710'), 'name': 'Omar', 'courses': ['Math', 'Physics', 'Chemistry']}


## 📋 Important Execution Notes

* 🟢 **MongoDB Service Status**: Ensure that the local MongoDB service is actively running on your Windows machine before executing any code cells.
* 🔢 **Execution Order**: Run all cells sequentially from top to bottom to maintain the proper flow of data and state management.
* 🧹 **Data Idempotency**: The very first notebook execution cell clears and resets the `students` collection. This prevents data duplication and keeps tests consistent upon multiple re-runs.
* 🛠️ **Troubleshooting Dependencies**: If you encounter a `ModuleNotFoundError: No module named 'pymongo'` exception, uncomment and execute the `!pip install pymongo` line located within the first configuration cell.